In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 0 — Data Audit & Universe Filtering
# ───────────────────────────────────────────────────────────────────────────────
#
# BUSINESS PURPOSE
# ────────────────
# We start with 500+ cryptocurrencies but most of them would corrupt our analysis
# if we included them. This phase is like cleaning a database before running
# reports — garbage in, garbage out.
#
# We apply 4 quality gates, one by one:
#   1. Does this coin have enough price history to be meaningful?
#   2. Is this actually a real crypto, not a stablecoin (like USDC)?
#   3. Is this coin actually traded, or is it a ghost with no real buyers?
#   4. Are there suspicious gaps in the price data?
#
# The goal: shrink 500 coins to a clean ~125–250 that we can trust.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

os.makedirs("outputs", exist_ok=True)

# ── Load raw data ──────────────────────────────────────────────────────────────
# This is your full dataset: ~500 coins × 2 years of daily price data
df = pd.read_csv("crypto_data.csv", parse_dates=["date"])
df = df.sort_values(["coin_id", "date"]).reset_index(drop=True)

print(f"Raw shape      : {df.shape}")
print(f"Coins          : {df['coin_id'].nunique()}")
print(f"Date range     : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Total days     : {df['date'].nunique()}")


# ═══════════════════════════════════════════════════════════════════════════════
# GATE 1 — COVERAGE CHECK
# ───────────────────────────────────────────────────────────────────────────────
#
# WHAT WE'RE DOING
# ────────────────
# We check how many days of price history each coin actually has.
# Our dataset covers ~730 days (2 years). We require at least 400 days.
#
# WHY THIS MATTERS (business terms)
# ───────────────────────────────────
# Imagine comparing two employees — one with 2 years of performance reviews
# and another hired 3 months ago. You can't fairly compare them.
# Same idea here: a coin launched mid-2024 doesn't have enough history to
# tell us whether it behaves like Bitcoin or not. Short history → unreliable
# similarity scores → wrong fund transfer decisions.
#
# WHAT GETS REMOVED
# ──────────────────
# Coins that were newly launched during our analysis window, or were
# delisted early (often coins that collapsed or were abandoned).
# ═══════════════════════════════════════════════════════════════════════════════

TOTAL_DAYS = df["date"].nunique()
MIN_DAYS   = 400   # Must have at least 400 out of ~730 trading days

coverage = (
    df.groupby("coin_id")["date"]
    .count()
    .rename("trading_days")
    .reset_index()
)
coverage["coverage_pct"]  = (coverage["trading_days"] / TOTAL_DAYS * 100).round(2)
coverage["pass_coverage"] = coverage["trading_days"] >= MIN_DAYS

coins_pass_coverage = coverage.loc[coverage["pass_coverage"], "coin_id"].tolist()

print(f"\n[Gate 1 — Coverage]")
print(f"  Minimum required days : {MIN_DAYS}")
print(f"  Coins passing         : {len(coins_pass_coverage)} / {df['coin_id'].nunique()}")
print(f"  Coins removed         : {df['coin_id'].nunique() - len(coins_pass_coverage)} (too short a history)")

# ── Chart: How many trading days does each coin have? ─────────────────────────
# Green = passes the 400-day threshold | Red = fails (too new or delisted)
fig_cov = px.histogram(
    coverage,
    x="trading_days",
    nbins=50,
    color="pass_coverage",
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
    labels={"trading_days": "Number of Trading Days Available", "pass_coverage": "Passes Quality Gate"},
    title=(
        "Gate 1 — How Much Price History Does Each Coin Have?<br>"
        "<sup>Green = enough history (400+ days) | Red = too short, excluded</sup>"
    ),
    template="plotly_dark"
)
fig_cov.add_vline(
    x=MIN_DAYS, line_dash="dash", line_color="white",
    annotation_text=f"Minimum required: {MIN_DAYS} days",
    annotation_position="top right"
)
fig_cov.update_layout(
    xaxis_title="Number of Trading Days Available",
    yaxis_title="Number of Coins",
    legend_title="Passes Quality Gate"
)
fig_cov.show()


# ═══════════════════════════════════════════════════════════════════════════════
# GATE 2 — STABLECOIN REMOVAL
# ───────────────────────────────────────────────────────────────────────────────
#
# WHAT WE'RE DOING
# ────────────────
# We remove stablecoins (USDC, USDT, etc.) and any coin that behaves like one.
#
# WHY THIS MATTERS (business terms)
# ───────────────────────────────────
# Stablecoins are pegged to $1. Their price never moves.
# If we left them in, our system would say "USDC and Bitcoin are 80% similar"
# simply because both have very stable correlations — but for completely
# different reasons. USDC is stable because it's designed to be $1.
# This would produce completely fake "redundancy" signals.
#
# The tricky part: some algorithmic stablecoins (like UST that collapsed in 2022)
# are NOT flagged in the database as stablecoins. We detect them manually by
# checking if their price barely moved over 2 years (coefficient of variation < 0.5%).
#
# WHAT GETS REMOVED
# ──────────────────
# Flagged stablecoins (USDT, USDC, DAI, etc.) + any "hidden" stablecoins
# detected by near-zero price movement.
# ═══════════════════════════════════════════════════════════════════════════════

df_f = df[df["coin_id"].isin(coins_pass_coverage)].copy()

# Remove coins explicitly tagged as stablecoins in the database
stable_ids = df_f.loc[df_f["is_stablecoin"] == True, "coin_id"].unique()
df_f       = df_f[df_f["is_stablecoin"] != True].copy()

# Detect "hidden" stablecoins: price barely moved over 2 years
# A real crypto should have at least 0.5% average price fluctuation relative to its mean.
# Anything below that is essentially a pegged coin.
price_stats = (
    df_f.groupby("coin_id")["close"]
    .agg(["mean", "std"])
    .assign(cv=lambda x: x["std"] / x["mean"])   # Coefficient of Variation
    .reset_index()
)
algo_stable_suspects = price_stats.loc[price_stats["cv"] < 0.005, "coin_id"].tolist()

if algo_stable_suspects:
    print(f"\n[Gate 2 — Stablecoins] ⚠ Hidden stablecoins detected: {algo_stable_suspects}")
    print(f"  These coins show near-zero price movement over 2 years.")
    print(f"  They are NOT flagged in the database but behave like pegged coins.")
    df_f = df_f[~df_f["coin_id"].isin(algo_stable_suspects)].copy()

coins_after_stable = df_f["coin_id"].nunique()
print(f"\n[Gate 2 — Stablecoins]")
print(f"  Removed (database-flagged stablecoins) : {len(stable_ids)}")
print(f"  Removed (hidden / algorithmic)         : {len(algo_stable_suspects)}")
print(f"  Coins remaining                        : {coins_after_stable}")


# ═══════════════════════════════════════════════════════════════════════════════
# GATE 3 — LIQUIDITY FILTER (Amihud Illiquidity Ratio)
# ───────────────────────────────────────────────────────────────────────────────
#
# WHAT WE'RE DOING
# ────────────────
# We measure how "tradeable" each coin actually is using the Amihud ratio:
#   Amihud = Average(|daily price move| / daily dollar volume traded)
#
# A HIGH Amihud ratio = small trades move the price a lot = illiquid = thin market
# A LOW  Amihud ratio = large trades barely move the price  = liquid  = healthy market
#
# We remove the 20% most illiquid coins AND any coin with median daily
# trading volume below $500,000 (basically nobody is trading it).
#
# WHY THIS MATTERS (business terms)
# ───────────────────────────────────
# Illiquid coins have flat price charts not because they're similar to other coins,
# but simply because nobody trades them. Their price sits still for days.
# This creates FAKE similarity signals — two illiquid coins look "correlated"
# because both do nothing, not because they actually move together.
#
# Practical example: If a fund tried to transfer $1M into an illiquid coin,
# their own buy order would move the market 10–20%. These coins are also
# extremely difficult to exit quickly. We don't want to flag them as
# "similar to Bitcoin" and recommend holding them instead of BTC.
#
# WHAT GETS REMOVED
# ──────────────────
# Small-cap coins with thin order books, micro-cap tokens, zombie projects
# that are technically listed but have almost no real trading activity.
# ═══════════════════════════════════════════════════════════════════════════════

df_f = df_f.sort_values(["coin_id", "date"])

# Compute daily log return for each coin
# Log returns are better than % returns for statistical analysis
df_f["log_return"] = df_f.groupby("coin_id")["close"].transform(
    lambda x: np.log(x / x.shift(1))
)

# Avoid division by zero on zero-volume days
df_f["volume_usd_safe"] = df_f["volume_usd"].replace(0, np.nan)

# Daily Amihud: how much does $1 of trading move the price?
df_f["amihud_daily"] = df_f["log_return"].abs() / df_f["volume_usd_safe"]

# Average over entire history per coin
amihud = (
    df_f.groupby("coin_id")
    .agg(
        amihud_ratio     = ("amihud_daily",    "mean"),
        median_daily_vol = ("volume_usd_safe", "median"),
    )
    .reset_index()
)

ILLIQ_PERCENTILE = 0.80    # Remove top 20% most illiquid
HARD_VOL_FLOOR   = 500_000 # Hard minimum: $500K median daily volume

illiq_threshold = amihud["amihud_ratio"].quantile(ILLIQ_PERCENTILE)

amihud["pass_amihud"] = (
    (amihud["amihud_ratio"]     <= illiq_threshold) &
    (amihud["median_daily_vol"] >= HARD_VOL_FLOOR)
)

coins_pass_liquidity = amihud.loc[amihud["pass_amihud"], "coin_id"].tolist()

print(f"\n[Gate 3 — Liquidity]")
print(f"  Illiquidity cutoff (80th percentile) : {illiq_threshold:.2e}")
print(f"  Hard volume minimum                  : ${HARD_VOL_FLOOR:,} / day")
print(f"  Coins passing                        : {len(coins_pass_liquidity)}")
print(f"  Coins removed (illiquid / thin)      : {coins_after_stable - len(coins_pass_liquidity)}")

# ── Chart 1: Distribution of illiquidity scores across all coins ──────────────
# Shows clearly how the illiquid tail (red) is separated from tradeable coins (green)
fig_amihud = px.histogram(
    amihud, x="amihud_ratio",
    color="pass_amihud", nbins=60, log_x=True,
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
    labels={
        "amihud_ratio": "Illiquidity Score (higher = harder to trade)",
        "pass_amihud":  "Passes Liquidity Gate"
    },
    title=(
        "Gate 3 — How Tradeable Is Each Coin? (Amihud Illiquidity Score)<br>"
        "<sup>Green = sufficiently liquid | Red = too illiquid, price data is unreliable</sup>"
    ),
    template="plotly_dark"
)
fig_amihud.add_vline(
    x=illiq_threshold, line_dash="dash", line_color="yellow",
    annotation_text="Liquidity cutoff (80th percentile)"
)
fig_amihud.update_layout(
    xaxis_title="Illiquidity Score — log scale (higher = harder to trade, price barely moves)",
    yaxis_title="Number of Coins"
)
fig_amihud.show()

# ── Chart 2: Volume vs Illiquidity — the full liquidity landscape ─────────────
# Each dot is one coin. Top-left = trouble zone (high volume but illiquid = suspicious).
# Bottom-right = ideal (high volume, very liquid = reliable price signal).
fig_liq_scatter = px.scatter(
    amihud, x="median_daily_vol", y="amihud_ratio",
    color="pass_amihud", log_x=True, log_y=True,
    hover_data=["coin_id"],
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
    labels={
        "median_daily_vol": "Median Daily Trading Volume (USD)",
        "amihud_ratio":     "Illiquidity Score"
    },
    title=(
        "Gate 3 — Liquidity Landscape: Trading Volume vs Illiquidity Score<br>"
        "<sup>Each dot = one coin. Green = passes. Red = excluded. "
        "Bottom-right corner = most liquid (best quality).</sup>"
    ),
    template="plotly_dark"
)
fig_liq_scatter.add_vline(
    x=HARD_VOL_FLOOR, line_dash="dash", line_color="orange",
    annotation_text="$500K minimum volume floor"
)
fig_liq_scatter.add_hline(
    y=illiq_threshold, line_dash="dash", line_color="yellow",
    annotation_text="Illiquidity cutoff"
)
fig_liq_scatter.update_layout(
    xaxis_title="Median Daily Trading Volume USD — log scale (right = more traded)",
    yaxis_title="Illiquidity Score — log scale (down = more liquid)"
)
fig_liq_scatter.show()


# ═══════════════════════════════════════════════════════════════════════════════
# GATE 4 — MISSING VALUE AUDIT (Price Gap Check)
# ───────────────────────────────────────────────────────────────────────────────
#
# WHAT WE'RE DOING
# ────────────────
# We check each coin's price history for gaps — days where no price was recorded.
#
# Small gaps (1–2 days): Normal. Exchange maintenance, public holidays, weekends
#   on some platforms. We fill these by carrying the last known price forward.
#
# Medium gaps (3–7 days): Suspicious. Could be exchange problems, trading halts,
#   or early data collection issues. We fill but flag with a warning.
#
# Large gaps (8+ days): A coin with 8+ consecutive missing days is not trustworthy.
#   We cannot reasonably "invent" 8 days of price data. These coins are excluded.
#
# WHY THIS MATTERS (business terms)
# ───────────────────────────────────
# Imagine a price chart with a straight horizontal line for 10 days — that's
# what a filled gap looks like. If our similarity algorithm sees two coins
# that both had flat lines for the same 10-day period (even by coincidence),
# it will declare them "highly correlated." That's a completely artificial signal.
# We either fix small gaps honestly, or cut coins with large holes in their data.
# ═══════════════════════════════════════════════════════════════════════════════

df_liq = df_f[df_f["coin_id"].isin(coins_pass_liquidity)].copy()

def max_consecutive_nans(series: pd.Series) -> int:
    """Find the longest streak of missing days in a price series."""
    mask = series.isna()
    if not mask.any():
        return 0
    groups = mask.ne(mask.shift()).cumsum()
    return int(mask.groupby(groups).sum().max())

gap_stats = (
    df_liq.groupby("coin_id")["close"]
    .apply(max_consecutive_nans)
    .rename("max_gap_days")
    .reset_index()
)

nan_pct = (
    df_liq.groupby("coin_id")[["close", "volume_usd"]]
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index()
)
nan_pct.columns = ["coin_id", "close_nan_pct", "volume_nan_pct"]
gap_stats = gap_stats.merge(nan_pct, on="coin_id")

def gap_action(row) -> str:
    if row["max_gap_days"] > 7:
        return "exclude"               # Too large to fill honestly
    elif row["max_gap_days"] >= 3:
        return "forward_fill_warn"     # Fill but flag it
    else:
        return "forward_fill_ok"       # Normal small gap, fill silently

gap_stats["action"] = gap_stats.apply(gap_action, axis=1)
coins_to_exclude    = gap_stats.loc[gap_stats["action"] == "exclude", "coin_id"].tolist()

print(f"\n[Gate 4 — Data Gaps]")
print(f"  Excluded (gap > 7 days, data unreliable) : {len(coins_to_exclude)}")
print(f"  Flagged  (gap 3–7 days, filled w/ warning): {(gap_stats['action']=='forward_fill_warn').sum()}")
print(f"  Clean    (gap ≤ 2 days, normal)           : {(gap_stats['action']=='forward_fill_ok').sum()}")

# ── Chart: Which coins have the worst data gaps? ──────────────────────────────
# Red = excluded. Orange = filled with warning. Green = clean data.
fig_gaps = px.bar(
    gap_stats.sort_values("max_gap_days", ascending=False).head(50),
    x="coin_id", y="max_gap_days",
    color="action",
    color_discrete_map={
        "exclude":           "#e74c3c",   # Red  = thrown out
        "forward_fill_warn": "#f39c12",   # Orange = kept but flagged
        "forward_fill_ok":   "#2ecc71"    # Green = clean
    },
    title=(
        "Gate 4 — Worst 50 Coins by Longest Consecutive Missing Days<br>"
        "<sup>Red = excluded (data too unreliable) | Orange = kept with warning | Green = clean</sup>"
    ),
    labels={
        "max_gap_days": "Longest Consecutive Gap (days)",
        "coin_id":      "Coin",
        "action":       "Decision"
    },
    template="plotly_dark"
)
fig_gaps.add_hline(
    y=7, line_dash="dash", line_color="red",
    annotation_text="Exclusion threshold: gaps > 7 days"
)
fig_gaps.add_hline(
    y=3, line_dash="dash", line_color="orange",
    annotation_text="Warning threshold: gaps ≥ 3 days"
)
fig_gaps.update_layout(
    xaxis_tickangle=45,
    xaxis_title="Coin",
    yaxis_title="Longest Consecutive Gap in Price Data (days)",
    legend_title="Data Quality Decision"
)
fig_gaps.show()


# ═══════════════════════════════════════════════════════════════════════════════
# FINAL UNIVERSE — Compile clean working dataset
# ───────────────────────────────────────────────────────────────────────────────
# All 4 gates applied. Remaining coins get small gaps filled (max 7 days).
# This is the dataset that flows into every downstream analysis phase.
# ═══════════════════════════════════════════════════════════════════════════════

final_coins = [c for c in coins_pass_liquidity if c not in coins_to_exclude]
df_universe = df_liq[df_liq["coin_id"].isin(final_coins)].copy()

# Fill small gaps (≤ 7 days) by carrying last known price forward
for col in ["close", "open", "high", "low", "volume_usd"]:
    df_universe[col] = df_universe.groupby("coin_id")[col].transform(
        lambda x: x.ffill(limit=7)
    )

print(f"\n{'═'*50}")
print(f"  PHASE 0 COMPLETE — Working Universe Summary")
print(f"{'═'*50}")
print(f"  Raw input           : {df['coin_id'].nunique():>5} coins")
print(f"  After Gate 1        : {len(coins_pass_coverage):>5} coins  (removed: short history)")
print(f"  After Gate 2        : {coins_after_stable:>5} coins  (removed: stablecoins)")
print(f"  After Gate 3        : {len(coins_pass_liquidity):>5} coins  (removed: illiquid/untradeable)")
print(f"  After Gate 4        : {len(final_coins):>5} coins  (removed: bad data gaps)")
print(f"{'─'*50}")
print(f"  ✅ WORKING UNIVERSE  : {len(final_coins):>5} coins  → ready for similarity analysis")

# ── Chart: The full filtering story in one picture ────────────────────────────
# A funnel shows how many coins survived each quality gate.
# This is the summary chart for stakeholders — simple, no math, just the story.
fig_funnel = go.Figure(go.Funnel(
    y=[
        "500+ Raw Coins (starting universe)",
        "Gate 1: Enough History (400+ days)",
        "Gate 2: Not a Stablecoin",
        "Gate 3: Actually Traded ($500K+/day)",
        "Gate 4: Clean Price Data"
    ],
    x=[
        df["coin_id"].nunique(),
        len(coins_pass_coverage),
        coins_after_stable,
        len(coins_pass_liquidity),
        len(final_coins)
    ],
    textinfo="value+percent initial",
    marker_color=["#3498db", "#9b59b6", "#e67e22", "#e74c3c", "#2ecc71"],
    textfont={"size": 14}
))
fig_funnel.update_layout(
    title={
        "text": (
            "Phase 0 — How We Built the Clean Working Universe<br>"
            "<sup>Each stage removes coins that would corrupt the similarity analysis. "
            "Only trustworthy, liquid, real-crypto data passes through.</sup>"
        ),
        "x": 0.5
    },
    template="plotly_dark",
    height=500
)
fig_funnel.show()


# ── Save outputs for downstream phases ────────────────────────────────────────
pd.DataFrame({"coin_id": final_coins}).to_csv(
    "outputs/01_universe_filtered.csv", index=False
)
(
    amihud[amihud["coin_id"].isin(final_coins)]
    .merge(gap_stats[["coin_id", "max_gap_days", "action"]], on="coin_id", how="left")
    .to_csv("outputs/03_coin_metadata_phase0.csv", index=False)
)
df_universe.to_parquet("outputs/phase0_clean_data.parquet", index=False)

print("\n✅ Saved: 01_universe_filtered.csv  → list of 125 clean coins")
print("✅ Saved: 03_coin_metadata_phase0.csv → liquidity scores + data quality per coin")
print("✅ Saved: phase0_clean_data.parquet   → clean OHLCV ready for Phase 0.5")

Raw shape      : (677378, 20)
Coins          : 732
Date range     : 1970-01-01 → 2026-04-17
Total days     : 1825

[Gate 1 — Coverage]
  Minimum required days : 400
  Coins passing         : 515 / 732
  Coins removed         : 217 (too short a history)



[Gate 2 — Stablecoins] ⚠ Hidden stablecoins detected: ['global-dollar', 'ring-usd']
  These coins show near-zero price movement over 2 years.
  They are NOT flagged in the database but behave like pegged coins.

[Gate 2 — Stablecoins]
  Removed (database-flagged stablecoins) : 12
  Removed (hidden / algorithmic)         : 2
  Coins remaining                        : 501

[Gate 3 — Liquidity]
  Illiquidity cutoff (80th percentile) : 4.54e-07
  Hard volume minimum                  : $500,000 / day
  Coins passing                        : 357
  Coins removed (illiquid / thin)      : 144



[Gate 4 — Data Gaps]
  Excluded (gap > 7 days, data unreliable) : 0
  Flagged  (gap 3–7 days, filled w/ warning): 0
  Clean    (gap ≤ 2 days, normal)           : 357



══════════════════════════════════════════════════
  PHASE 0 COMPLETE — Working Universe Summary
══════════════════════════════════════════════════
  Raw input           :   732 coins
  After Gate 1        :   515 coins  (removed: short history)
  After Gate 2        :   501 coins  (removed: stablecoins)
  After Gate 3        :   357 coins  (removed: illiquid/untradeable)
  After Gate 4        :   357 coins  (removed: bad data gaps)
──────────────────────────────────────────────────
  ✅ WORKING UNIVERSE  :   357 coins  → ready for similarity analysis



✅ Saved: 01_universe_filtered.csv  → list of 125 clean coins
✅ Saved: 03_coin_metadata_phase0.csv → liquidity scores + data quality per coin
✅ Saved: phase0_clean_data.parquet   → clean OHLCV ready for Phase 0.5
